# Fuzzy-CapsNet — Brain Tumor Classification

**Hybrid Fuzzy Capsule Network** for classifying brain MRI scans into 4 categories:
- **Glioma** · **Meningioma** · **No Tumor** · **Pituitary**

---
##   Section 1 — Imports & Configuration

In [ ]:
import os
import sys
import time
import argparse
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm import tqdm
from PIL import Image

import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    accuracy_score, roc_auc_score,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
print(' All imports successful')

 All imports successful


In [ ]:
# Configuration 
IMG_SIZE             = 64        
BATCH_SIZE           = 16      
NUM_CLASSES          = 4
NUM_EPOCHS           = 50
LEARNING_RATE        = 1e-3
WEIGHT_DECAY         = 1e-4      # L2 regularization
DROPOUT_P            = 0.3
EARLY_STOP_PATIENCE  = 5
VAL_SPLIT            = 0.15      # 15% of training set used for validation
ROUTING_ITERS        = 3         # fuzzy routing iterations

DATA_ROOT    = './dataset'
MODEL_PATH   = 'fuzzy_capsnet.pt'
EVAL_OUT_DIR = 'eval_output'
os.makedirs(EVAL_OUT_DIR, exist_ok=True)

CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

# Device 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    torch.set_num_threads(4)
print(f'  Device : {DEVICE}')

#  Dark-theme plot style 
DARK_BG   = '#0D1117'
DARK_SURF = '#161B22'
DARK_CARD = '#1C2128'
GRID_COL  = '#21262D'
TEXT_PRI  = '#E6EDF3'
TEXT_SEC  = '#8B949E'
ACCENT    = '#58A6FF'
PALETTE   = ['#58A6FF', '#3FB950', '#F85149', '#E3B341']

plt.rcParams.update({
    'figure.facecolor':   DARK_BG,
    'axes.facecolor':     DARK_SURF,
    'axes.edgecolor':     GRID_COL,
    'axes.grid':          True,
    'grid.color':         GRID_COL,
    'grid.linewidth':     0.7,
    'axes.labelcolor':    TEXT_SEC,
    'xtick.color':        TEXT_SEC,
    'ytick.color':        TEXT_SEC,
    'text.color':         TEXT_PRI,
    'font.family':        'DejaVu Sans',
    'font.size':          11,
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'savefig.dpi':        150,
    'savefig.bbox':       'tight',
    'savefig.facecolor':  DARK_BG,
    'legend.facecolor':   DARK_CARD,
    'legend.edgecolor':   GRID_COL,
    'legend.labelcolor':  TEXT_PRI,
})

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
print(' Configuration done')

  Device : cpu
 Configuration done


---
##  Section 2 — Model Architecture

```
ConvFeatureExtractor  →  PrimaryCapsLayer  →  FuzzyCapsuleLayer  →  L2-norm  →  Softmax
```

**Core innovation:** Gaussian Membership routing replaces the hard argmax used in classic CapsNets:

$$\mu(\hat{u}, v) = \exp\!\left(-\frac{\|\hat{u} - v\|^2}{2\sigma^2}\right)$$

In [ ]:
#  Squash activation 
def squash(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """Squash vector to length < 1 while preserving direction."""
    norm_sq = (x ** 2).sum(dim=dim, keepdim=True)
    norm    = norm_sq.sqrt()
    scale   = norm_sq / (1.0 + norm_sq)
    return scale * (x / (norm + 1e-8))


#  Layer 1: Convolutional Feature Extractor 
class ConvFeatureExtractor(nn.Module):
    """3-block CNN: raw pixels → rich feature maps."""

    def __init__(self, in_channels: int = 3, dropout_p: float = DROPOUT_P):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.dropout = nn.Dropout(p=dropout_p)

    def forward(self, x):
        return self.dropout(self.block3(self.block2(self.block1(x))))


#  Layer 2: Primary Capsule Layer 
class PrimaryCapsLayer(nn.Module):
    """CNN feature maps → 8-D capsule vectors."""

    def __init__(self, in_channels: int = 128,
                 capsule_dim: int = 8, num_capsules: int = 32):
        super().__init__()
        self.capsule_dim  = capsule_dim
        self.num_capsules = num_capsules
        self.conv = nn.Conv2d(in_channels, num_capsules * capsule_dim,
                              kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        out = self.conv(x)
        B, _, H, W = out.shape
        out = out.view(B, self.num_capsules, self.capsule_dim, H * W)
        out = out.permute(0, 1, 3, 2).contiguous()
        out = out.view(B, -1, self.capsule_dim)
        return squash(out)


#  Layer 3: Fuzzy Capsule Layer (Core innovation) 
class FuzzyCapsuleLayer(nn.Module):
    """
    Routing-by-Fuzzy-Agreement.
    Gaussian membership replaces hard coupling coefficients:
        μ(u_hat, v) = exp( -‖u_hat − v‖² / (2σ²) )
    Benefits:
      • Naturally bounded in [0,1] — interpretable as fuzzy membership
      • Smooth gradient flow
      • Handles intra-class variation gracefully
    """

    def __init__(self, num_in_capsules, num_out_capsules=NUM_CLASSES,
                 in_dim=8, out_dim=16, routing_iters=ROUTING_ITERS):
        super().__init__()
        self.num_in  = num_in_capsules
        self.num_out = num_out_capsules
        self.iters   = routing_iters
        # Transformation matrices W_{ij}
        self.W = nn.Parameter(
            torch.randn(1, num_in_capsules, num_out_capsules, out_dim, in_dim) * 0.01)
        # Learnable σ per output capsule
        self.log_sigma = nn.Parameter(torch.ones(num_out_capsules) * 0.5)

    def forward(self, u):
        B = u.size(0)
        u_exp = u.unsqueeze(2).unsqueeze(-1)
        W_exp = self.W.expand(B, -1, -1, -1, -1)
        u_hat = torch.matmul(W_exp, u_exp).squeeze(-1)   # (B, Ni, No, Do)

        b = torch.zeros(B, self.num_in, self.num_out,
                        device=u.device, dtype=u.dtype)
        v = None
        for iteration in range(self.iters):
            c = F.softmax(b, dim=2)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=1)
            v = squash(s)
            if iteration == self.iters - 1:
                break
            sigma    = self.log_sigma.exp().clamp(min=1e-4)
            v_exp    = v.unsqueeze(1).expand_as(u_hat)
            diff_sq  = ((u_hat - v_exp) ** 2).sum(dim=-1)
            sigma_sq = (sigma ** 2).unsqueeze(0).unsqueeze(0)
            membership = torch.exp(-diff_sq / (2 * sigma_sq + 1e-8))
            b = b + membership
        return v


#  Full Model 
class FuzzyCapsNet(nn.Module):
    """Hybrid Fuzzy Capsule Network for Brain Tumor Classification."""

    def __init__(self, img_size=IMG_SIZE, in_channels=3,
                 num_classes=NUM_CLASSES, dropout_p=DROPOUT_P,
                 routing_iters=ROUTING_ITERS):
        super().__init__()
        feat_spatial     = img_size // 8
        num_primary      = 32 * (feat_spatial ** 2)
        self.conv_extractor = ConvFeatureExtractor(in_channels, dropout_p)
        self.primary_caps   = PrimaryCapsLayer(128, 8, 32)
        self.fuzzy_caps     = FuzzyCapsuleLayer(num_primary, num_classes,
                                                8, 16, routing_iters)
        self.dropout        = nn.Dropout(p=dropout_p)
        self.num_classes    = num_classes

    def forward(self, x):
        """Returns (norms, norms) — L2 norm of each class capsule."""
        features = self.conv_extractor(x)
        u        = self.primary_caps(features)
        u        = self.dropout(u)
        v        = self.fuzzy_caps(u)
        norms    = v.norm(dim=-1)      # (B, num_classes)  in (0,1)
        return norms, norms

    def predict_proba(self, x):
        """Returns (softmax_probs, fuzzy_membership_scores)."""
        self.eval()
        with torch.no_grad():
            scores, norms = self(x)
        probs      = F.softmax(norms, dim=-1)
        norm_min   = norms.min(dim=-1, keepdim=True).values
        norm_max   = norms.max(dim=-1, keepdim=True).values
        membership = (norms - norm_min) / (norm_max - norm_min + 1e-8)
        return probs, membership


# Quick sanity check
model = FuzzyCapsNet(img_size=IMG_SIZE).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
dummy    = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
norms, _ = model(dummy)
print(f' Model built — {n_params:,} trainable parameters')
print(f'   Output shape: {norms.shape}  (batch=2, classes={NUM_CLASSES})')

 Model built — 1,437,444 trainable parameters
   Output shape: torch.Size([2, 4])  (batch=2, classes=4)


---
##  Section 3 — Data Loading

In [ ]:
def get_transforms(img_size: int = IMG_SIZE, augment: bool = True):
    norm = transforms.Normalize(mean=MEAN, std=STD)
    if augment:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                   saturation=0.1, hue=0.05),
            transforms.ToTensor(), norm,
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(), norm,
    ])


def load_datasets(data_root=DATA_ROOT, img_size=IMG_SIZE,
                  batch_size=BATCH_SIZE, val_split=VAL_SPLIT):
    train_dir = os.path.join(data_root, 'Training')
    test_dir  = os.path.join(data_root, 'Testing')

    full_train = datasets.ImageFolder(train_dir,
                     transform=get_transforms(img_size, augment=True))
    test_ds    = datasets.ImageFolder(test_dir,
                     transform=get_transforms(img_size, augment=False))

    n_val   = int(len(full_train) * val_split)
    n_train = len(full_train) - n_val
    train_ds, val_ds = random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(42))
    val_ds.dataset.transform = get_transforms(img_size, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=0)

    print(f'Dataset: {data_root}')
    print(f'  Classes : {full_train.classes}')
    print(f'  Train   : {n_train}   Val: {n_val}   Test: {len(test_ds)}')
    return train_loader, val_loader, test_loader, full_train.classes


train_loader, val_loader, test_loader, class_names = load_datasets()

Dataset: ./dataset
  Classes : ['glioma', 'meningioma', 'notumor', 'pituitary']
  Train   : 4760   Val: 840   Test: 1600


---
## Section 4 — Training

In [ ]:
#  Loss Function 
class MarginLoss(nn.Module):
    """CapsNet Margin Loss (Hinton et al. 2017)."""
    def __init__(self, m_plus=0.9, m_minus=0.1, lam=0.5):
        super().__init__()
        self.m_plus  = m_plus
        self.m_minus = m_minus
        self.lam     = lam

    def forward(self, norms, labels):
        T     = F.one_hot(labels, num_classes=norms.size(1)).float()
        left  = F.relu(self.m_plus  - norms) ** 2
        right = F.relu(norms - self.m_minus) ** 2
        loss  = T * left + self.lam * (1 - T) * right
        return loss.sum(dim=1).mean()


#  Early Stopping 
class EarlyStopping:
    def __init__(self, patience=EARLY_STOP_PATIENCE, delta=1e-4):
        self.patience   = patience
        self.delta      = delta
        self.counter    = 0
        self.best_loss  = None
        self.stop       = False
        self.best_state = None

    def __call__(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.delta:
            self.best_loss  = val_loss
            self.counter    = 0
            self.best_state = {k: v.cpu().clone()
                               for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True


#  Training helpers 
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, leave=False, desc='  train', ncols=80)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        norms, _ = model(imgs)
        loss = criterion(norms, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds   = norms.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        pbar.set_postfix(loss=f'{loss.item():.3f}', acc=f'{correct/total:.3f}')
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        norms, _ = model(imgs)
        loss = criterion(norms, labels)
        total_loss += loss.item() * imgs.size(0)
        preds   = norms.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader,
                num_epochs=NUM_EPOCHS, lr=LEARNING_RATE,
                wd=WEIGHT_DECAY, device=DEVICE):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3)
    criterion = MarginLoss()
    stopper   = EarlyStopping(patience=EARLY_STOP_PATIENCE)
    history   = {'train_loss': [], 'val_loss': [],
                 'train_acc':  [], 'val_acc':  []}

    print(f'\n{"="*55}')
    print(f'  Training on {device}  |  max {num_epochs} epochs')
    print(f'{"="*55}')

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                          optimizer, criterion, device)
        vl_loss, vl_acc = evaluate_loader(model, val_loader, criterion, device)
        scheduler.step(vl_loss)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        print(f'  Epoch {epoch:3d}/{num_epochs} '
              f'| tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} '
              f'| vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} '
              f'| {time.time()-t0:.1f}s')

        stopper(vl_loss, model)
        if stopper.stop:
            print(f'\n  ⏹  Early stopping at epoch {epoch}.')
            break

    if stopper.best_state:
        model.load_state_dict(stopper.best_state)
        print(f'  ✔  Best weights restored (val_loss={stopper.best_loss:.4f})')

    return history


print('Training helpers defined')

Training helpers defined


In [6]:
# Run Training 
model   = FuzzyCapsNet(img_size=IMG_SIZE).to(DEVICE)
history = train_model(model, train_loader, val_loader, device=DEVICE)

torch.save(model.state_dict(), MODEL_PATH)
print(f'\n  Model saved → {MODEL_PATH}')


  Training on cpu  |  max 50 epochs


  Epoch   1/50 | tr_loss=0.2482 tr_acc=0.6977 | vl_loss=0.1624 vl_acc=0.7774 | 199.2s


  Epoch   2/50 | tr_loss=0.1766 tr_acc=0.8126 | vl_loss=0.1493 vl_acc=0.8512 | 227.2s


  Epoch   3/50 | tr_loss=0.1483 tr_acc=0.8475 | vl_loss=0.1484 vl_acc=0.8417 | 189.2s


  Epoch   4/50 | tr_loss=0.1329 tr_acc=0.8697 | vl_loss=0.1100 vl_acc=0.8679 | 182.7s


...

  Epoch  46/50 | tr_loss=0.0073 tr_acc=1.0000 | vl_loss=0.0342 vl_acc=0.9595 | 253.8s


  Epoch  47/50 | tr_loss=0.0070 tr_acc=0.9998 | vl_loss=0.0370 vl_acc=0.9583 | 279.7s


  Epoch  48/50 | tr_loss=0.0074 tr_acc=0.9996 | vl_loss=0.0357 vl_acc=0.9631 | 310.3s


  Epoch  49/50 | tr_loss=0.0070 tr_acc=0.9996 | vl_loss=0.0349 vl_acc=0.9607 | 347.8s


  Epoch  50/50 | tr_loss=0.0068 tr_acc=1.0000 | vl_loss=0.0367 vl_acc=0.9571 | 390.2s
  ✔  Best weights restored (val_loss=0.0342)

  Model saved → fuzzy_capsnet.pt


In [7]:
# Training Curves 
def plot_training_curves(history, save_path='training_curves.png'):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(epochs, history['train_loss'], color='#58a6ff', lw=2, label='Train')
    axes[0].plot(epochs, history['val_loss'],   color='#f78166', lw=2,
                 label='Val', linestyle='--')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Margin Loss')
    axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], color='#3fb950', lw=2, label='Train')
    axes[1].plot(epochs, history['val_acc'],   color='#d29922', lw=2,
                 label='Val', linestyle='--')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy'); axes[1].set_ylim(0, 1)
    axes[1].legend()

    fig.suptitle('Fuzzy-CapsNet  |  Training Curves',
                 fontsize=15, fontweight='bold', color=TEXT_PRI, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path); plt.close()
    print(f' Saved → {save_path}')


plot_training_curves(history)

 Saved → training_curves.png


---
## Section 5 — Quick Evaluation (Metrics)

Accuracy, Precision, Recall, F1, ROC-AUC on both Train & Test sets.

In [8]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        norms, _ = model(imgs)
        probs = F.softmax(norms, dim=-1).cpu().numpy()
        preds = norms.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.numpy())
        all_probs.append(probs)
    return (np.concatenate(all_labels),
            np.concatenate(all_preds),
            np.concatenate(all_probs))


def print_metrics(y_true, y_pred, y_probs, split_name):
    acc      = accuracy_score(y_true, y_pred)
    prec     = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec      = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_w     = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    try:
        y_bin     = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
        auc_score = roc_auc_score(y_bin, y_probs,
                                  multi_class='ovr', average='macro')
        auc_str = f'{auc_score:.4f}'
    except Exception:
        auc_str = 'N/A'

    div = '═' * 55
    print(f'\n{div}')
    print(f'{split_name} SET METRICS')
    print(div)
    print(f'  Accuracy          : {acc:.4f}  ({acc*100:.2f}%)')
    print(f'  Precision (wt.)   : {prec:.4f}')
    print(f'  Recall    (wt.)   : {rec:.4f}')
    print(f'  F1-Score  (wt.)   : {f1_w:.4f}')
    print(f'  F1-Score  (macro) : {f1_macro:.4f}')
    print(f'  ROC-AUC   (macro) : {auc_str}')
    print()
    print(classification_report(y_true, y_pred,
                                target_names=CLASS_NAMES,
                                digits=4, zero_division=0))


# Collect & print 
# Load train without augmentation for fair metric measurement
train_dir_path = os.path.join(DATA_ROOT, 'Training')
test_dir_path  = os.path.join(DATA_ROOT, 'Testing')

train_eval_ds = datasets.ImageFolder(train_dir_path,
                    transform=get_transforms(IMG_SIZE, augment=False))
test_eval_ds  = datasets.ImageFolder(test_dir_path,
                    transform=get_transforms(IMG_SIZE, augment=False))
train_eval_loader = DataLoader(train_eval_ds, batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=0)
test_eval_loader  = DataLoader(test_eval_ds,  batch_size=BATCH_SIZE,
                               shuffle=False, num_workers=0)

print(' Running inference on Training set ...')
tr_true, tr_pred, tr_probs = collect_predictions(model, train_eval_loader, DEVICE)
print(' Running inference on Test set ...')
te_true, te_pred, te_probs = collect_predictions(model, test_eval_loader, DEVICE)

print_metrics(tr_true, tr_pred, tr_probs, 'TRAINING')
print_metrics(te_true, te_pred, te_probs, 'TEST')

 Running inference on Training set ...
 Running inference on Test set ...

═══════════════════════════════════════════════════════
TRAINING SET METRICS
═══════════════════════════════════════════════════════
  Accuracy          : 0.9939  (99.39%)
  Precision (wt.)   : 0.9939
  Recall    (wt.)   : 0.9939
  F1-Score  (wt.)   : 0.9939
  F1-Score  (macro) : 0.9939
  ROC-AUC   (macro) : 0.9997

              precision    recall  f1-score   support

      Glioma     0.9936    0.9907    0.9921      1400
  Meningioma     0.9886    0.9893    0.9889      1400
    No Tumor     0.9979    0.9957    0.9968      1400
   Pituitary     0.9957    1.0000    0.9979      1400

    accuracy                         0.9939      5600
   macro avg     0.9939    0.9939    0.9939      5600
weighted avg     0.9939    0.9939    0.9939      5600


═══════════════════════════════════════════════════════
TEST SET METRICS
═══════════════════════════════════════════════════════
  Accuracy          : 0.9150  (91.50%)
  P

In [9]:
# Confusion Matrix + Per-Class Metrics + ROC-AUC 
def _styled_ax(ax):
    ax.set_facecolor(DARK_SURF)
    ax.tick_params(colors=TEXT_PRI)
    ax.xaxis.label.set_color(TEXT_SEC)
    ax.yaxis.label.set_color(TEXT_SEC)
    ax.title.set_color(TEXT_PRI)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID_COL)


def plot_cm_quick(y_true, y_pred, save_path='confusion_matrix.png'):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5, linecolor=GRID_COL)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix  —  Test Set')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right', color=TEXT_PRI)
    plt.setp(ax.get_yticklabels(), color=TEXT_PRI)
    plt.tight_layout(); plt.savefig(save_path); plt.close()
    print(f' Saved → {save_path}')


def plot_per_class_metrics(y_true, y_pred, save_path='per_class_metrics.png'):
    report  = classification_report(y_true, y_pred,
                  target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    metrics = ['precision', 'recall', 'f1-score']
    x, width = np.arange(len(CLASS_NAMES)), 0.25
    colors   = ['#58a6ff', '#3fb950', '#d29922']
    fig, ax  = plt.subplots(figsize=(10, 5))
    _styled_ax(ax)
    for i, (m, c) in enumerate(zip(metrics, colors)):
        vals = [report[cls][m] for cls in CLASS_NAMES]
        bars = ax.bar(x + i*width, vals, width, label=m.capitalize(),
                      color=c, alpha=0.85, edgecolor=GRID_COL)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
                    f'{v:.2f}', ha='center', va='bottom',
                    color=TEXT_PRI, fontsize=8)
    ax.set_xticks(x + width); ax.set_xticklabels(CLASS_NAMES, rotation=15, ha='right')
    ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
    ax.set_title('Per-Class Metrics: Precision / Recall / F1')
    ax.legend()
    plt.tight_layout(); plt.savefig(save_path); plt.close()
    print(f' Saved → {save_path}')


def plot_roc_auc(y_true, y_probs, save_path='roc_auc.png'):
    from sklearn.metrics import roc_curve, auc
    y_bin    = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    fig, ax  = plt.subplots(figsize=(8, 6))
    _styled_ax(ax)
    for i, (cls, color) in enumerate(zip(CLASS_NAMES, PALETTE)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        ax.plot(fpr, tpr, color=color, lw=2,
                label=f'{cls}  (AUC={auc(fpr,tpr):.3f})')
    ax.plot([0,1],[0,1], color=GRID_COL, linestyle='--', lw=1.2)
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves — One-vs-Rest'); ax.legend(fontsize=9)
    plt.tight_layout(); plt.savefig(save_path); plt.close()
    print(f' Saved → {save_path}')


plot_cm_quick(te_true, te_pred)
plot_per_class_metrics(te_true, te_pred)
plot_roc_auc(te_true, te_probs)

 Saved → confusion_matrix.png
 Saved → per_class_metrics.png
 Saved → roc_auc.png


---
## Section 6 — Full Evaluation & Visualizations

Generates 8 diagnostic plots saved to `outputs/`:

| # | Plot |
|---|------|
| 1 | Class Distribution |
| 2 | Sample Images |
| 3 | Preprocessing Pipeline |
| 4 | Confusion Matrix (styled) |
| 5 | Per-Class Accuracy |
| 6 | Confidence Distribution |
| 7 | Fuzzy Membership Scores |
| 8 | Misclassified Samples |

In [10]:
def _save(fig, name):
    path = os.path.join(EVAL_OUT_DIR, name)
    fig.savefig(path); plt.close(fig)
    print(f'  saved → {name}')

def denormalize(tensor):
    t = tensor.clone()
    for c, m, s in zip(range(3), MEAN, STD):
        t[c] = t[c] * s + m
    return t.permute(1, 2, 0).clamp(0, 1).numpy()


# Plot 01: Class Distribution 
def plot_class_distribution(train_dir, test_dir):
    def counts(d):
        return [len(os.listdir(os.path.join(d, c)))
                for c in sorted(os.listdir(d))
                if os.path.isdir(os.path.join(d, c))]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Dataset — Class Distribution',
                 fontsize=15, fontweight='bold', color=TEXT_PRI, y=1.02)
    for ax, cnts, title, alpha in zip(
            axes, [counts(train_dir), counts(test_dir)],
            ['Training Set', 'Test Set'], [1.0, 0.75]):
        bars = ax.bar(CLASS_NAMES, cnts, color=PALETTE, alpha=alpha,
                      width=0.55, edgecolor=DARK_BG, linewidth=1.2)
        for bar, v in zip(bars, cnts):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(cnts)*0.02,
                    str(v), ha='center', fontsize=11,
                    fontweight='bold', color=TEXT_PRI)
        ax.set_title(title, color=TEXT_PRI, pad=10)
        ax.set_ylabel('Image Count', color=TEXT_SEC)
        ax.set_ylim(0, max(cnts)*1.22)
        ax.set_xticklabels(CLASS_NAMES, rotation=15, ha='right')
    fig.tight_layout(); _save(fig, '01_class_distribution.png')


# Plot 02: Sample Images 
def plot_sample_images(train_dir):
    classes = sorted([d for d in os.listdir(train_dir)
                      if os.path.isdir(os.path.join(train_dir, d))])
    fig, axes = plt.subplots(4, 5, figsize=(15, 12))
    fig.suptitle('Sample Images — 5 per Class',
                 fontsize=15, fontweight='bold', color=TEXT_PRI, y=1.02)
    for row, cls in enumerate(classes):
        cls_dir = os.path.join(train_dir, cls)
        for col, fname in enumerate(sorted(os.listdir(cls_dir))[:5]):
            img = Image.open(os.path.join(cls_dir, fname)).convert('RGB')
            ax  = axes[row][col]
            ax.imshow(img); ax.axis('off')
            if col == 0:
                ax.set_ylabel(CLASS_NAMES[row], fontsize=12,
                              fontweight='bold', color=PALETTE[row])
    fig.tight_layout(); _save(fig, '02_sample_images.png')


# Plot 03: Preprocessing Pipeline 
def plot_preprocessing(train_dir, img_size=IMG_SIZE):
    cls_dirs = sorted([d for d in os.listdir(train_dir)
                       if os.path.isdir(os.path.join(train_dir, d))])
    sample_path = os.path.join(
        train_dir, cls_dirs[0],
        sorted(os.listdir(os.path.join(train_dir, cls_dirs[0])))[0])
    orig = Image.open(sample_path).convert('RGB')
    steps = [
        ('Original',       transforms.Resize((img_size, img_size))),
        ('Random Flip',    transforms.Compose([transforms.Resize((img_size, img_size)),
                                               transforms.RandomHorizontalFlip(p=1.0)])),
        ('Rotation ±15',   transforms.Compose([transforms.Resize((img_size, img_size)),
                                               transforms.RandomRotation(15)])),
        ('Color Jitter',   transforms.Compose([transforms.Resize((img_size, img_size)),
                                               transforms.ColorJitter(0.3,0.3,0.2,0.1)])),
        ('Normalized',     transforms.Compose([transforms.Resize((img_size, img_size)),
                                               transforms.ToTensor(),
                                               transforms.Normalize(MEAN, STD)])),
    ]
    fig, axes = plt.subplots(1, 5, figsize=(17, 4))
    fig.suptitle('Preprocessing & Data Augmentation Pipeline',
                 fontsize=15, fontweight='bold', color=TEXT_PRI, y=1.02)
    for ax, (title, tfm) in zip(axes, steps):
        out = tfm(orig)
        if isinstance(out, torch.Tensor):
            out = denormalize(out)
        ax.imshow(out); ax.set_title(title, color=TEXT_PRI, fontsize=10, pad=8)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor(ACCENT); spine.set_linewidth(1.2); spine.set_visible(True)
    fig.tight_layout(); _save(fig, '03_preprocessing.png')


# Plot 04: Styled Confusion Matrix 
def plot_confusion_matrix_styled(y_true, y_pred):
    cm  = confusion_matrix(y_true, y_pred)
    acc = np.diag(cm).sum() / cm.sum()
    cmap = LinearSegmentedColormap.from_list(
        'dark_blue', [DARK_SURF, '#0D419D', '#58A6FF'], N=256)
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(cm, cmap=cmap, aspect='auto')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_SEC)
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=20, ha='right', color=TEXT_PRI)
    ax.set_yticklabels(CLASS_NAMES, color=TEXT_PRI)
    ax.set_xlabel('Predicted', labelpad=10, color=TEXT_SEC)
    ax.set_ylabel('Ground Truth', labelpad=10, color=TEXT_SEC)
    ax.set_title(f'Confusion Matrix  |  Test Accuracy = {acc*100:.2f}%',
                 pad=14, color=TEXT_PRI)
    thresh = cm.max() / 2
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    fontsize=13, fontweight='bold',
                    color=TEXT_PRI if cm[i,j] <= thresh else DARK_BG)
    fig.tight_layout(); _save(fig, '04_confusion_matrix.png')
    return cm


# Plot 05: Per-Class Accuracy 
def plot_per_class_accuracy(cm):
    per_class = np.diag(cm) / cm.sum(axis=1)
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(CLASS_NAMES, per_class*100,
                   color=PALETTE, edgecolor=DARK_BG, linewidth=1.2, height=0.45)
    for bar, v in zip(bars, per_class):
        ax.text(v*100+0.8, bar.get_y()+bar.get_height()/2,
                f'{v*100:.1f}%', va='center', fontsize=11,
                fontweight='bold', color=TEXT_PRI)
    ax.set_xlim(0, 115)
    ax.set_xlabel('Accuracy (%)', color=TEXT_SEC)
    ax.set_title('Per-Class Accuracy — Test Set', color=TEXT_PRI, pad=12)
    ax.axvline(90, color='#F85149', linestyle='--', lw=1.2, alpha=0.8,
               label='90% threshold')
    ax.legend(fontsize=9); fig.tight_layout(); _save(fig, '05_per_class_accuracy.png')


# Plot 06: Confidence Distribution 
@torch.no_grad()
def get_full_predictions(model, loader, device):
    model.eval()
    y_true, y_pred, confidences, all_norms = [], [], [], []
    all_imgs, all_labels = [], []
    for imgs, labels in loader:
        norms, _ = model(imgs.to(device))
        probs = F.softmax(norms, dim=-1)
        preds = norms.argmax(dim=1)
        conf  = probs.max(dim=1).values
        y_true.extend(labels.tolist())
        y_pred.extend(preds.cpu().tolist())
        confidences.extend(conf.cpu().tolist())
        all_norms.extend(norms.cpu().tolist())
        all_imgs.extend(imgs)
        all_labels.extend(labels.tolist())
    return (np.array(y_true), np.array(y_pred),
            np.array(confidences), np.array(all_norms),
            all_imgs, all_labels)

def plot_confidence_distribution(y_true, y_pred, confidences):
    correct = confidences[y_true == y_pred]
    wrong   = confidences[y_true != y_pred]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('Confidence Score Distribution',
                 fontsize=15, fontweight='bold', color=TEXT_PRI, y=1.02)
    ax = axes[0]
    ax.hist(correct, bins=22, color='#3FB950', alpha=0.85,
            edgecolor=DARK_BG, label=f'Correct ({len(correct)})')
    ax.hist(wrong,   bins=22, color='#F85149', alpha=0.85,
            edgecolor=DARK_BG, label=f'Wrong ({len(wrong)})')
    ax.axvline(confidences.mean(), color=ACCENT, linestyle='--', lw=1.5,
               label=f'Mean={confidences.mean():.2f}')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Count')
    ax.set_title('Correct vs Wrong Predictions'); ax.legend()
    ax2 = axes[1]
    for i, cls in enumerate(CLASS_NAMES):
        mask = y_true == i
        if mask.sum() > 0:
            ax2.hist(confidences[mask], bins=18, alpha=0.7,
                     color=PALETTE[i], edgecolor=DARK_BG, label=cls)
    ax2.set_xlabel('Confidence'); ax2.set_ylabel('Count')
    ax2.set_title('Confidence by Class'); ax2.legend(fontsize=9)
    fig.tight_layout(); _save(fig, '06_confidence_distribution.png')


# Plot 07: Fuzzy Membership Scores 
def plot_fuzzy_membership(all_imgs, all_labels, all_norms):
    indices = [next((i for i, l in enumerate(all_labels) if l == cls), None)
               for cls in range(len(CLASS_NAMES))]
    indices = [i for i in indices if i is not None]
    fig = plt.figure(figsize=(14, len(indices)*3.0))
    fig.suptitle('Fuzzy Membership Scores — One Sample per Class',
                 fontsize=15, fontweight='bold', color=TEXT_PRI, y=1.02)
    for row, idx in enumerate(indices):
        norms      = np.array(all_norms[idx])
        label      = all_labels[idx]
        membership = (norms - norms.min()) / (norms.max() - norms.min() + 1e-8)
        ax_img = fig.add_subplot(len(indices), 2, row*2+1)
        ax_img.imshow(denormalize(all_imgs[idx]))
        ax_img.set_title(f'True: {CLASS_NAMES[label]}',
                         fontsize=10, color=PALETTE[label], fontweight='bold', pad=6)
        ax_img.axis('off')
        ax_bar = fig.add_subplot(len(indices), 2, row*2+2)
        colors = [PALETTE[i] if membership[i] == membership.max()
                  else '#30363D' for i in range(len(CLASS_NAMES))]
        bars = ax_bar.barh(CLASS_NAMES, membership, color=colors,
                           edgecolor=DARK_BG, linewidth=0.8, height=0.45)
        for bar, v in zip(bars, membership):
            ax_bar.text(v+0.015, bar.get_y()+bar.get_height()/2,
                        f'{v:.3f}', va='center', fontsize=10, color=TEXT_PRI)
        ax_bar.set_xlim(0, 1.3)
        ax_bar.set_xlabel('Membership Score', color=TEXT_SEC)
        ax_bar.set_title('Fuzzy Membership', color=TEXT_PRI, fontsize=10)
    fig.tight_layout(); _save(fig, '07_fuzzy_membership.png')


# Plot 08: Misclassified Samples 
def plot_misclassified(all_imgs, all_labels, y_true, y_pred, confidences, n=12):
    wrong_idx = np.where(y_true != y_pred)[0][:n]
    if len(wrong_idx) == 0:
        print('Perfect test set — no misclassified samples!'); return
    cols = 4; rows = (len(wrong_idx) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, rows*3.8))
    fig.suptitle('Misclassified Samples',
                 fontsize=15, fontweight='bold', color=TEXT_PRI, y=1.02)
    axes = axes.flatten()
    for ax, idx in zip(axes, wrong_idx):
        ax.imshow(denormalize(all_imgs[idx]))
        ax.set_title(
            f'True: {CLASS_NAMES[y_true[idx]]}\n'
            f'Pred: {CLASS_NAMES[y_pred[idx]]}  ({confidences[idx]*100:.1f}%)',
            fontsize=9, color='#F85149', pad=6)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor('#F85149'); spine.set_linewidth(1.5)
            spine.set_visible(True)
    for ax in axes[len(wrong_idx):]:
        ax.set_visible(False)
    fig.tight_layout(); _save(fig, '08_misclassified.png')


print('Evaluation plot functions defined')

Evaluation plot functions defined


In [11]:
# Run all 8 plots 
print('Generating all evaluation plots ...\n')

y_true_full, y_pred_full, confidences, all_norms, all_imgs, all_labels = \
    get_full_predictions(model, test_eval_loader, DEVICE)

plot_class_distribution(train_dir_path, test_dir_path)
plot_sample_images(train_dir_path)
plot_preprocessing(train_dir_path)
cm = plot_confusion_matrix_styled(y_true_full, y_pred_full)
plot_per_class_accuracy(cm)
plot_confidence_distribution(y_true_full, y_pred_full, confidences)
plot_fuzzy_membership(all_imgs, all_labels, all_norms)
plot_misclassified(all_imgs, all_labels, y_true_full, y_pred_full, confidences)

print(f'\n All 8 plots saved to  →  {EVAL_OUT_DIR}/')

Generating all evaluation plots ...

  saved → 01_class_distribution.png
  saved → 02_sample_images.png
  saved → 03_preprocessing.png
  saved → 04_confusion_matrix.png
  saved → 05_per_class_accuracy.png
  saved → 06_confidence_distribution.png
  saved → 07_fuzzy_membership.png
  saved → 08_misclassified.png

 All 8 plots saved to  →  eval_output/


---
## Section 7 — Single-Image Prediction & Explainability

Shows the input image alongside **Fuzzy Membership Scores** and **Softmax Probabilities** — key for explainability.

In [12]:
def predict_and_explain(model, image_tensor, class_names=CLASS_NAMES,
                        save_path='prediction_explanation.png'):
    """
    Given a single preprocessed image tensor (1, C, H, W), display:
      1. The input image
      2. Fuzzy Membership Scores (0–1)  ← explainability
      3. Softmax probabilities
    """
    probs_t, membership_t = model.predict_proba(image_tensor)
    probs_np  = probs_t[0].numpy()
    memb_np   = membership_t[0].numpy()
    pred_idx  = probs_np.argmax()
    pred_class = class_names[pred_idx]
    confidence = probs_np[pred_idx] * 100

    fig = plt.figure(figsize=(14, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

    # Image
    ax_img = fig.add_subplot(gs[0])
    img_show = image_tensor[0].permute(1, 2, 0).numpy()
    img_show = img_show * np.array(STD) + np.array(MEAN)
    ax_img.imshow(np.clip(img_show, 0, 1))
    ax_img.set_title('Input Image', color=TEXT_PRI, fontsize=12,
                     fontweight='bold'); ax_img.axis('off')

    # Fuzzy Membership
    ax_fuzz = fig.add_subplot(gs[1])
    colors  = [PALETTE[pred_idx] if i == pred_idx else '#484f58'
               for i in range(NUM_CLASSES)]
    bars = ax_fuzz.barh(class_names, memb_np, color=colors, height=0.5)
    ax_fuzz.set_xlim(0, 1)
    ax_fuzz.set_title('Fuzzy Membership Scores', color=TEXT_PRI,
                      fontsize=12, fontweight='bold')
    ax_fuzz.set_xlabel('Degree (0=Non-member, 1=Full member)',
                       color=TEXT_SEC, fontsize=8)
    for bar, val in zip(bars, memb_np):
        ax_fuzz.text(min(val+0.02, 0.93),
                     bar.get_y()+bar.get_height()/2,
                     f'{val:.3f}', va='center', color=TEXT_PRI, fontsize=9)

    # Softmax Probs
    ax_prob = fig.add_subplot(gs[2])
    colors2 = [PALETTE[pred_idx] if i == pred_idx else '#484f58'
               for i in range(NUM_CLASSES)]
    bars2 = ax_prob.barh(class_names, probs_np*100, color=colors2, height=0.5)
    ax_prob.set_xlim(0, 100)
    ax_prob.set_title('Softmax Probabilities (%)', color=TEXT_PRI,
                      fontsize=12, fontweight='bold')
    ax_prob.set_xlabel('Confidence (%)', color=TEXT_SEC, fontsize=8)
    for bar, val in zip(bars2, probs_np*100):
        ax_prob.text(min(val+1, 88),
                     bar.get_y()+bar.get_height()/2,
                     f'{val:.1f}%', va='center', color=TEXT_PRI, fontsize=9)

    fig.suptitle(f'Prediction: {pred_class}  |  Confidence: {confidence:.1f}%',
                 fontsize=14, fontweight='bold', color=ACCENT, y=1.03)
    plt.savefig(save_path); plt.close()
    print(f' Saved → {save_path}')

    # Console summary
    print(f'\n{"─"*50}')
    print(f'  Predicted : {pred_class}   ({confidence:.2f}%)')
    print(f'  Fuzzy Membership Scores:')
    for cls, score in zip(class_names, memb_np):
        bar = '█' * int(score*20)
        print(f'    {cls:<15} {score:.4f}  {bar}')
    print(f'{"─"*50}\n')
    return pred_class, probs_np, memb_np


# Demo on first test image 
sample_imgs, sample_labels = next(iter(test_eval_loader))
pred_class, probs, membership = predict_and_explain(
    model, sample_imgs[0:1], save_path='prediction_explanation.png')

 Saved → prediction_explanation.png

──────────────────────────────────────────────────
  Predicted : Meningioma   (33.34%)
  Fuzzy Membership Scores:
    Glioma          0.0000  
    Meningioma      1.0000  ████████████████████
    No Tumor        0.8864  █████████████████
    Pituitary       0.0881  █
──────────────────────────────────────────────────

